In [1]:
%load_ext autoreload
%autoreload 2

# `Logit` on Orders - Logistic Regression (~1h)

## Select features

🎯 Haydi `wait_time` ve `delay_vs_expected` değişkenlerinin çok `iyi/kötü review`lar üzerindeki etkisini inceleyelim.

👉 `orders` training_set’imizi kullanarak iki adet `multivariate logistic regression` çalıştıracağız:
- `logit_one` → `dim_is_one_star` tahmini için  
- `logit_five` → `dim_is_five_star` tahmini için.

 

In [2]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

👉 Dataset’inizi import edin:

In [3]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)

In [4]:
orders.head()

,order_id,wait_time,expected_wait_time,delay_vs_expected,order_status,dim_is_five_star,dim_is_one_star,review_score,number_of_items,number_of_sellers,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,15.544063,0.0,delivered,0,0,4,1,1,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,19.137766,0.0,delivered,0,0,4,1,1,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,26.639711,0.0,delivered,1,0,5,1,1,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,26.188819,0.0,delivered,1,0,5,1,1,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,12.112049,0.0,delivered,1,0,5,1,1,19.90,8.72


In [7]:
orders.info()

<class 'pandas.DataFrame'>
Index: 96353 entries, 0 to 96360
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            96353 non-null  str    
 1   wait_time           96353 non-null  float64
 2   expected_wait_time  96353 non-null  float64
 3   delay_vs_expected   96353 non-null  float64
 4   order_status        96353 non-null  str    
 5   dim_is_five_star    96353 non-null  int64  
 6   dim_is_one_star     96353 non-null  int64  
 7   review_score        96353 non-null  int64  
 8   number_of_items     96353 non-null  int64  
 9   number_of_sellers   96353 non-null  int64  
 10  price               96353 non-null  float64
 11  freight_value       96353 non-null  float64
dtypes: float64(5), int64(5), str(2)
memory usage: 13.3 MB


👉 Kullanmak istediğiniz feature’ları bir listede seçin:

⚠️ Data leakage yaratmadığınızdan emin olun (yani target’tan türetilmiş feature’ları seçmeyin)

💡 `wait_time` ve `delay_vs_expected` değişkenlerinin etkisini anlayabilmek için diğer feature’ların etkisini kontrol etmemiz gerekir, bu yüzden listenize ilgili olabilecek tüm feature’ları dahil edin.

In [10]:
# SENİN KODUN BURAYA
features = ['wait_time', 'expected_wait_time', 'delay_vs_expected', 'number_of_items', 'number_of_sellers', 'price', 'freight_value']

🕵🏻 Feature’larınızın `multicollinearity` durumunu `VIF index` kullanarak kontrol edin.

* Çok yüksek olmamalıdır (tercihen < 10), böylece partial regression coefficient’larına ve ilgili `p-values` değerlerine güvenebiliriz.
* Verinizi standardize etmeyi unutmayın!
    * Bir `VIF Analysis`, bir feature’ın diğer feature’lara karşı regresyonunu yaparak hesaplanır...
    * Bu yüzden herhangi bir linear regression çalıştırmadan önce feature’ların `scale etkisini kaldırmak` ve eşit öneme sahip olmalarını sağlamak istersiniz!
    
    
📚 <a href="https://www.statisticshowto.com/variance-inflation-factor/">Statistics How To - Variance Inflation Factor</a>

📚  <a href="https://online.stat.psu.edu/stat462/node/180/">PennState - Detecting Multicollinearity Using Variance Inflation Factors</a>

⚖️ Standardize etme:

In [12]:
# SENİN KODUN BURAYA
orders_standardized = orders.copy()

for f in features:
    mu = orders_standardized[f].mean()
    sigma = orders_standardized[f].std()
    orders_standardized[f] = (orders_standardized[f] - mu) / sigma


In [13]:
orders_standardized[features].describe().round(2)

,wait_time,expected_wait_time,delay_vs_expected,number_of_items,number_of_sellers,price,freight_value
count,96353.00,96353.00,96353.00,96353.00,96353.00,96353.00,96353.00
mean,-0.00,0.00,0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.27,-2.48,-0.16,-0.26,-0.11,-0.65,-1.06
25%,-0.61,-0.62,-0.16,-0.26,-0.11,-0.44,-0.41
50%,-0.24,-0.06,-0.16,-0.26,-0.11,-0.24,-0.26
75%,0.33,0.53,-0.16,-0.26,-0.11,0.06,0.06
max,20.71,15.00,40.42,37.09,32.25,64.13,82.44


👉 Olası multicollinearity durumlarını analiz etmek için VIF Analysis’inizi çalıştırın:

In [ ]:
#X = orders_standardized[features].values

In [ ]:
#X.shape[1]

7

In [17]:
# SENİN KODUN BURAYA
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = orders_standardized[features].values

vif_values = []

for i in range(X.shape[1]):
    vif_values.append(variance_inflation_factor(X, i))

vif_df = pd.DataFrame({
    'feature': features,
    'vif': vif_values
}).sort_values(by='vif', ascending=False)

vif_df

,feature,vif
0,wait_time,2.832725
2,delay_vs_expected,2.379695
6,freight_value,1.577027
1,expected_wait_time,1.443633
3,number_of_items,1.348236
5,price,1.204880
4,number_of_sellers,1.095442


## Logistic Regressions

👉 İki adet `Logistic Regression` modeli fit edin:
- `logit_one` → `dim_is_one_star` tahmini için
- `logit_five` → `dim_is_five_star` tahmini için.

`Logit 1️⃣`

In [18]:
# SENİN KODUN BURAYA
logit_one = smf.logit(formula='dim_is_one_star ~ wait_time + expected_wait_time + delay_vs_expected + number_of_items + number_of_sellers + price + freight_value', data=orders_standardized).fit()

Optimization terminated successfully.
         Current function value: 0.273079
         Iterations 7


In [20]:
logit_one.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        dim_is_one_star   No. Observations:                96353
Model:                          Logit   Df Residuals:                    96345
Method:                           MLE   Df Model:                            7
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                  0.1461
Time:                        12:34:17   Log-Likelihood:                -26312.
converged:                       True   LL-Null:                       -30814.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -2.4824      0.013   -189.580      0.000      -2.508      -2.457
wait_time              0.7956      0.019     41.530      0.000       0.758       0.833
expected_wait_time    -0.2484      0.016    -15.450      0.000      -0.280      -0.217
delay_vs_expected      0.1573      0.020      7.794      0.000       0.118       0.197
number_of_items        0.2645      0.011     24.977      0.000       0.244       0.285
number_of_sellers      0.1867      0.008     23.639      0.000       0.171       0.202
price                  0.0579      0.011      5.127      0.000       0.036       0.080
freight_value         -0.0422      0.013     -3.193      0.001      -0.068      -0.016
======================================================================================
"""

`Logit 5️⃣`

In [19]:
# SENİN KODUN BURAYA
logit_five = smf.logit(formula='dim_is_five_star ~ wait_time + expected_wait_time + delay_vs_expected + number_of_items + number_of_sellers + price + freight_value', data=orders_standardized).fit()

Optimization terminated successfully.
         Current function value: 0.636930
         Iterations 7


In [21]:
logit_five.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:       dim_is_five_star   No. Observations:                96353
Model:                          Logit   Df Residuals:                    96345
Method:                           MLE   Df Model:                            7
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.05787
Time:                        12:34:28   Log-Likelihood:                -61370.
converged:                       True   LL-Null:                       -65140.
Covariance Type:            nonrobust   LLR p-value:                     0.000
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3427      0.007     48.102      0.000       0.329       0.357
wait_time             -0.5389      0.012    -44.095      0.000      -0.563      -0.515
expected_wait_time     0.0943      0.008     11.206      0.000       0.078       0.111
delay_vs_expected     -0.3818      0.024    -16.046      0.000      -0.428      -0.335
number_of_items       -0.1447      0.008    -17.454      0.000      -0.161      -0.128
number_of_sellers     -0.1450      0.008    -18.550      0.000      -0.160      -0.130
price                  0.0167      0.008      2.201      0.028       0.002       0.032
freight_value          0.0180      0.009      2.071      0.038       0.001       0.035
======================================================================================
"""

## 📊 Sonuçların Yorumu

### 1. Feature seçimi ve multicollinearity
- Kullanılan feature'lar: `wait_time`, `expected_wait_time`, `delay_vs_expected`, `number_of_items`, `number_of_sellers`, `price`, `freight_value`.
- Data leakage olmaması için hedeften türetilen sütunlar (`review_score`, `dim_is_one_star`, `dim_is_five_star`) modele alınmadı. `distance_seller_customer` ise opsiyonel olduğu ve hesaplanmadığı için dahil edilmedi.
- Feature'lar z-score ile standardize edildi. Böylece katsayılar "1 standart sapma artış" cinsinden okunabiliyor ve birbiriyle karşılaştırılabiliyor.
- Tüm VIF değerleri < 3. Multicollinearity sorunu yok, katsayılara ve p-value'lara güvenebiliriz. `delay_vs_expected` 0'da kırpıldığı (`max(0, wait - expected)`) için üç zaman değişkeni arasında tam doğrusal bir bağ oluşmuyor.

### 2. Modellerin genel performansı
| | logit_one | logit_five |
|---|---|---|
| Pseudo R² | 0.146 | 0.058 |
| LLR p-value | 0.000 | 0.000 |

- İki model de istatistiksel olarak anlamlı.
- Bu feature'lar 1 yıldızı, 5 yıldızdan daha iyi açıklıyor. 5 yıldız alıp almamak modelde olmayan başka faktörlere (ürün kalitesi, beklenti vb.) daha çok bağlı.
- Tüm feature'lar ortalama değerindeyken 1 yıldız olasılığı ≈ %7.7, 5 yıldız olasılığı ≈ %58.

### 3. Katsayıların işaretleri ve anlamlılığı
- Tüm katsayılar p < 0.05 ile anlamlı. `price` ve `freight_value` logit_five'da sınırda anlamlı ve etkileri çok küçük.
- İşaretler iki modelde tutarlı biçimde ters:
  - `wait_time` ↑ → 1 yıldız olasılığı ↑, 5 yıldız olasılığı ↓
  - `delay_vs_expected` ↑ → 1 yıldız olasılığı ↑, 5 yıldız olasılığı ↓
  - `expected_wait_time` ↑ → 1 yıldız ↓, 5 yıldız ↑. Gerçek süre sabitken, beklenen süre uzunsa sipariş "erken" gelmiş sayılır.
  - `number_of_items` ve `number_of_sellers` ↑ → 1 yıldız ↑, 5 yıldız ↓. Çok ürün ve çok satıcı, parçalı teslimat ve daha fazla sorun ihtimali demek.

### 4. Teslimat süresi ve gecikmenin etkisi
| | logit_one | logit_five |
|---|---|---|
| `wait_time` | **+0.796** | −0.539 |
| `delay_vs_expected` | +0.157 | **−0.382** |

- **Gecikme**, 5 yıldız kaybında 1 yıldız kazanımından yaklaşık 2.4 kat daha etkili:
  - Delay 1 std arttığında 1 yıldız odds'u ×1.17 (%17 artış).
  - Delay 1 std arttığında 5 yıldız odds'u ×0.68 (%32 düşüş).
- **Toplam bekleme süresi** ise 1 yıldızı daha güçlü etkiliyor.
- **Sonuç:** Gecikme müşteriyi 5 yıldızdan uzaklaştırıyor, ama onu 1 yıldıza kadar götürmek için tek başına yeterli değil. 1 yıldızlar büyük olasılıkla teslimattan çok **ürünün kendisiyle ilgili sorunlardan** kaynaklanıyor (hasarlı, yanlış ya da kalitesiz ürün).
- ⚠️ İki modelin katsayılarını karşılaştırırken dikkatli olmak gerekiyor. 1 yıldız verenler ile 5 yıldız verenler farklı davranış kalıplarına sahip alt popülasyonlar olabilir.

### 5. OLS ile karşılaştırma
- OLS (`review_score`) katsayılarının işaretleri **logit_five ile aynı**, **logit_one'ın tersi** yönde. Bu beklenen bir sonuç, çünkü yüksek skor memnuniyet anlamına geliyor.
- OLS bize skordaki **ortalama** değişimi veriyor. Logit ise **uç durumların** (en kötü ve en iyi yorum) olasılığını modelliyor. Ödevin amacı olan "memnuniyetsizliği net yakalamak" için logit daha doğrudan bir cevap veriyor.

### 6. Logistic Regression neden sınıflandırma için daha uygun?
1. **Tahminler 0–1 arasında kalır.** Sigmoid fonksiyonu sayesinde çıktı her zaman geçerli bir olasılıktır. OLS ise 0'ın altında ya da 1'in üstünde anlamsız tahminler üretebilir.
2. **Doğrusal olmayan etki.** Olasılık S eğrisi boyunca değişir ve uçlarda doyuma ulaşır. OLS ise her birim artışın etkisini sabit varsayar.
3. **Varsayımlar.** Binary hedefte OLS'in hata terimleri normal dağılmaz ve varyansları sabit değildir (heteroskedasticity). Bu durumda p-value'lar güvenilmez olur. Logit ise MLE ile binary veriye uygun şekilde tahmin edilir.
4. **Yorumlanabilirlik.** Katsayılar log-odds üzerinde doğrusaldır. $e^{\beta}$ ile hesaplanan odds ratio, etkinin büyüklüğünü sezgisel bir şekilde ifade eder.

💡 Şimdi bu iki logistic regression’ın sonuçlarını analiz etme zamanı:

- Partial coefficient’ları kendi kelimelerinizle yorumlayın.
- `p-values` kullanarak istatistiksel anlamlılıklarını kontrol edin.
- Coefficient önemleri açısından `logit_one` ve `logit_five` arasında herhangi bir fark görüyor musunuz?

In [22]:
# Aşağıdaki cümlelerden doğru olanları aşağıdaki listeye kaydedin.

a = "delay_vs_expected influences five_star ratings even more than one_star ratings"
b = "wait_time influences five_star ratings even more than one_star"

your_answer = [a]

🧪 __Kodunu Test Et__

In [23]:
from nbresult import ChallengeResult

result = ChallengeResult('logit',
    answers = your_answer
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-9.0.3, pluggy-1.6.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/simgecevik/Desktop/WT/data-logit/tests
plugins: dash-4.4.1, anyio-4.2.0
collecting ... collected 1 item

test_logit.py::TestLogit::test_question PASSED                           [100%]

============================== 1 passed in 0.95s ===============================


💯 You can commit your code:

git add tests/logit.pickle

git commit -m 'Completed logit step'

git push origin master



<details>
    <summary>- <i>Açıklamalar ve ileri seviye kavramlar</i> -</summary>


> _Diğer tüm şeyler sabitken, `delay factor`, 1-yıldız review alma ihtimalini etkilemesinden bile daha fazla, 5-yıldızdan mahrum kalma ihtimalini artırma eğilimindedir. Muhtemelen bunun sebebi, 1-yıldız review’ların bizzat çok kötü ürünleri hedeflemesi, kötü teslimatları değil._

❗️ Ancak tamamen titiz olmak için, **iki farklı modelin coefficient’larını karşılaştırırken daha dikkatli olmamız gerekir**, çünkü **benzer popülasyonlara dayanmayabilirler**!
    Burada 2 alt popülasyonumuz var: (1-yıldız verenler ve 5-yıldız verenler) ve bunlar doğaları gereği farklı davranış kalıpları sergileyebilirler. 5-yıldız vermeye daha meyilli “mutlu insanlar”ın, “gecikme” veya “fiyat” söz konusu olduğunda, 1-yıldızı “Lucky-Luke gibi ateşleyen” “huysuz insanlara” göre daha az hassas olmaları gayet mümkün...

</details>



🏁 Tebrikler!

💾 `logit.ipynb` notebook’unuzu commit ve push etmeyi unutmayın!